# 📊 03. Post-QC, MultiQC Dashboard & Benchmark Analysis
### Comparative Analysis: RAW FASTQ vs Clean FASTQ

#### Key Objectives:
1. Quantify quality gains ($Q30$, Poly-A/G removal, adapter cleanup).
2. Compare Knee Plot curves before and after barcode error-correction.
3. Review interactive **MultiQC** dashboard.
4. Verify downstream alignment readiness for **STARsolo / Kallisto-Bustools / CellRanger**.


In [ ]:
import os
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import gzip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import IFrame, display, HTML

from src.utils import load_config
from src.sc_qc import SingleCellQC

config = load_config("../config/pipeline_config.yaml")
paths = config["paths"]


## 1. Compute Post-Clean QC Metrics

In [ ]:
# Run QC on Clean FASTQ
qc_clean = SingleCellQC(
    r1_path=f"../{paths['clean_r1']}",
    r2_path=f"../{paths['clean_r2']}",
    cb_len=config['chemistry']['r1_structure']['cell_barcode_len'],
    umi_len=config['chemistry']['r1_structure']['umi_len'],
    whitelist_path=f"../{paths['whitelist_file']}",
    sample_id="clean_sample_01"
)
clean_results = qc_clean.analyze_fastq()

# Load raw QC report
with open(f"../{paths['qc_raw_dir']}/{config['project']['sample_id']}_raw_sc_qc.json", "r") as f:
    raw_saved = json.load(f)
raw_m = raw_saved["summary_metrics"]
clean_m = clean_results["summary_metrics"]


## 2. Before vs After Quality Benchmark Table

In [ ]:
comparison_df = pd.DataFrame([
    {"Metric": "Total Reads", "RAW FASTQ": f"{raw_m['total_reads']:,}", "CLEAN FASTQ": f"{clean_m['total_reads']:,}", "Change": f"{(clean_m['total_reads']-raw_m['total_reads'])/raw_m['total_reads']*100:.1f}%"},
    {"Metric": "cDNA (R2) Q30 Rate", "RAW FASTQ": f"{raw_m['r2_q30_fraction']*100:.2f}%", "CLEAN FASTQ": f"{clean_m['r2_q30_fraction']*100:.2f}%", "Change": f"+{(clean_m['r2_q30_fraction']-raw_m['r2_q30_fraction'])*100:.2f}%"},
    {"Metric": "Barcode Q30 Rate", "RAW FASTQ": f"{raw_m['cb_q30_fraction']*100:.2f}%", "CLEAN FASTQ": f"{clean_m['cb_q30_fraction']*100:.2f}%", "Change": f"{(clean_m['cb_q30_fraction']-raw_m['cb_q30_fraction'])*100:.2f}%"},
    {"Metric": "Valid Barcodes", "RAW FASTQ": f"{raw_m.get('valid_barcode_fraction',0)*100:.2f}%", "CLEAN FASTQ": f"{clean_m.get('valid_barcode_fraction',0)*100:.2f}%", "Change": "100% Whitelist"},
    {"Metric": "Poly-A Artifact Rate", "RAW FASTQ": f"{raw_m['poly_a_rate']*100:.2f}%", "CLEAN FASTQ": f"{clean_m['poly_a_rate']*100:.2f}%", "Change": "Eliminated"},
    {"Metric": "Poly-G Artifact Rate", "RAW FASTQ": f"{raw_m['poly_g_rate']*100:.2f}%", "CLEAN FASTQ": f"{clean_m['poly_g_rate']*100:.2f}%", "Change": "Eliminated"}
])
display(comparison_df.style.set_properties(**{'text-align': 'left'}))


## 3. Comparative Visualizations

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Q30 Comparison Barplot
metrics_names = ["cDNA Q30", "CB Q30", "UMI Q30"]
raw_vals = [raw_m['r2_q30_fraction']*100, raw_m['cb_q30_fraction']*100, raw_m['umi_q30_fraction']*100]
clean_vals = [clean_m['r2_q30_fraction']*100, clean_m['cb_q30_fraction']*100, clean_m['umi_q30_fraction']*100]

x = np.arange(len(metrics_names))
width = 0.35

ax1.bar(x - width/2, raw_vals, width, label='Raw FASTQ', color='#90a4ae')
ax1.bar(x + width/2, clean_vals, width, label='Clean FASTQ', color='#2e7d32')
ax1.set_ylabel('Q30 Percentage (%)')
ax1.set_title('Phred Quality (Q30) Before vs After Cleaning', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names)
ax1.set_ylim(0, 110)
ax1.legend()

# Knee Plot Comparison
raw_counts = np.array(raw_saved["top_100_barcode_counts"])
clean_counts = np.array(clean_results["barcode_rank_counts"][:100])

ax2.plot(range(1, len(raw_counts)+1), raw_counts, label='Raw (with uncorrected)', color='#f57c00', lw=2)
ax2.plot(range(1, len(clean_counts)+1), clean_counts, label='Clean (Recovered & Corrected)', color='#1976d2', lw=2)
ax2.set_title('Knee Plot Comparison (Top Barcodes)', fontweight='bold')
ax2.set_xlabel('Barcode Rank')
ax2.set_ylabel('Read Depth')
ax2.legend()

plt.tight_layout()
plt.show()


## 4. MultiQC Aggregated Quality Dashboard
MultiQC compiles FastQC, Fastp, and tool metrics into an interactive HTML report.


In [ ]:
multiqc_path = Path("../reports/multiqc/single_cell_multiqc_report.html")
if multiqc_path.exists():
    print(f"✔ MultiQC report ready at: {multiqc_path.resolve()}")
    display(HTML(f'<p>👉 Open the interactive report: <a href="../{paths["multiqc_dir"]}/single_cell_multiqc_report.html" target="_blank" style="color: #1976d2; font-weight: bold;">MultiQC HTML Report</a></p>'))
else:
    print("MultiQC report not found. Run `pixi run multiqc-report` to generate it.")


## 5. Downstream Alignment Compatibility
The output FASTQ files are ready for standard single-cell aligners:
- **STARsolo**:
  ```bash
  STAR --genomeDir /path/to/ref --readFilesIn data/clean/scRNA_sample_01_val_R2.fastq.gz data/clean/scRNA_sample_01_val_R1.fastq.gz --soloType CB_UMI_Simple --soloCBwhitelist data/whitelist/737K-august-2016.txt
  ```
- **Kallisto / Bustools**:
  ```bash
  kallisto bus -i transcripts.idx -o bus_output/ -x 10xv3 -t 4 data/clean/scRNA_sample_01_val_R1.fastq.gz data/clean/scRNA_sample_01_val_R2.fastq.gz
  ```
- **Alevin / Salmon / UMI-tools**:
  Compatible with extracted header format in `data/clean/scRNA_sample_01_extracted.fastq.gz`.
